In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis: Tb2TiO7, HEiDi

This tutorial demonstrates a practical two-stage workflow for single-crystal
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with BUMPS-DREAM.

The example uses constant-wavelength neutron single-crystal diffraction data
for Tb2TiO7 measured on HEiDi at FRM II.

The goal is not only to obtain a good fit, but also to answer Bayesian
questions such as:

- Which parameter values are most probable?
- How broad are the credible intervals?
- Which parameters are strongly correlated?
- How much uncertainty propagates into the calculated reflection
  intensities?

## Import Library

In [2]:
import easydiffraction as ed

## Step 1: Create a Project Container

The project object keeps structures, experiments, fit settings, and
plotting utilities together in a single place. We will build the full
workflow inside this object.

In [3]:
project = ed.Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 2: Build the Structural Model

For this example we start from a CIF file describing the Tb2TiO7
pyrochlore structure. Loading the structure from CIF is convenient
because it preserves a realistic starting
model without rebuilding the full structure by hand.

In [4]:
structure_path = ed.download_data(id=20, destination='data')

Getting data...


Data #20: Tb2Ti2O7 (crystal structure)


✅ Data #20 already present at '/home/runner/work/diffraction-lib/diffraction-lib/data/ed-20.cif'. Keeping existing file.


In [5]:
project.structures.add_from_cif_path(structure_path)

In [6]:
structure = project.structures['tbti']

## Step 3: Define the Diffraction Experiment

Next we download the measured reflection data, create a neutron
single-crystal experiment, and configure the crystal link,
wavelength, and extinction model.

In [7]:
data_path = ed.download_data(id=19, destination='data')

Getting data...


Data #19: Tb2Ti2O7, HEiDi (MLZ)


✅ Data #19 already present at '/home/runner/work/diffraction-lib/diffraction-lib/data/ed-19.xye'. Keeping existing file.


In [8]:
project.experiments.add_from_data_path(
    name='heidi',
    data_path=data_path,
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'heidi'. Number of data points: 220.


In [9]:
experiment = project.experiments['heidi']

Link the crystal structure to the experiment and set its scale factor.

In [10]:
experiment.linked_crystal.id = 'tbti'
experiment.linked_crystal.scale = 1.0

Set the instrument wavelength and starting extinction parameters.
These values provide the initial experiment description for the local
refinement.

In [11]:
experiment.instrument.setup_wavelength = 0.793

In [12]:
experiment.extinction.mosaicity = 35000
experiment.extinction.radius = 10

## Step 4: Run an Initial Local Refinement

Before Bayesian sampling, it is useful to run a deterministic fit. This
gives us:

- a good point estimate near the best-fit region,
- uncertainties from the local optimizer,
- a quick check that the model and experiment are configured
  sensibly.

In this tutorial we refine a small set of structural and extinction
parameters while keeping occupancies fixed.

In [13]:
structure.atom_sites['O1'].fract_x.free = True

structure.atom_sites['Ti'].occupancy.free = False
structure.atom_sites['O1'].occupancy.free = False
structure.atom_sites['O2'].occupancy.free = False

structure.atom_sites['Tb'].adp_iso.free = True
structure.atom_sites['Ti'].adp_iso.free = True
structure.atom_sites['O1'].adp_iso.free = True
structure.atom_sites['O2'].adp_iso.free = True

In [14]:
experiment.linked_crystal.scale.free = True
experiment.extinction.radius.free = True

We keep using the default LMFIT Levenberg-Marquardt minimizer as a fast local
optimizer. Its main purpose here is to provide a stable starting point
and uncertainty estimates for the Bayesian run.

In [15]:
project.analysis.minimizer.show_supported()

Minimizer types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [16]:
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'heidi' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.29,592.02,
2,11,0.65,191.19,67.7% ↓
3,19,0.97,36.84,80.7% ↓
4,29,1.33,18.99,48.4% ↓
5,37,1.65,12.74,32.9% ↓
6,62,2.23,12.71,


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🏆 Best goodness-of-fit (reduced χ²) is 12.71 at iteration 61


✅ Fitting complete.


The fit-results display summarizes the locally refined values and their
estimated uncertainties.

In [17]:
project.display.fit.results()

Fit results


✅ Success: True


⏱️ Fitting time: 2.23 seconds


📏 Goodness-of-fit (reduced χ²): 12.71


📏 R-factor (Rf): 7.67%


...

📏 Weighted R-factor (wR): 8.52%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,start,fitted,uncertainty,change
1,tbti,atom_site,Tb,adp_iso,Å²,0.5300,0.5319,0.0190,0.36 % ↑
2,tbti,atom_site,Ti,adp_iso,Å²,0.4800,0.4776,0.0311,0.50 % ↓
3,tbti,atom_site,O1,fract_x,,0.3280,0.3280,0.0001,0.00 % ↓
4,tbti,atom_site,O1,adp_iso,Å²,0.4500,0.4504,0.0165,0.09 % ↑
5,tbti,atom_site,O2,adp_iso,Å²,0.2300,0.2375,0.0259,3.25 % ↑
6,heidi,extinction,,radius,μm,10.0000,26.4607,1.0003,164.61 % ↑
7,heidi,linked_crystal,,scale,,1.0000,2.9236,0.0488,192.36 % ↑


The correlation plot shows how strongly the refined parameters move
together in the local refinement. The measured-vs-calculated plot shows
how well the refined crystal model reproduces the measured reflection
intensities.

In [18]:
project.display.fit.correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
project.display.pattern(expt_name='heidi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5: Prepare for Bayesian Sampling

DREAM requires finite bounds for the free parameters. Instead of
setting them manually, we derive them from the uncertainties estimated
in the local refinement.

The helper method `set_fit_bounds_from_uncertainty` centers the bounds
on the current parameter value and expands them by a chosen multiple of
the reported uncertainty.

The default `multiplier` is 4. In this single-crystal tutorial we use
a tighter value of `1.5` to keep the sampling window closer to the
locally refined solution.

Show unset fit bounds before setting them from the local refinement
uncertainties.

In [20]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,tbti,atom_site,Tb,adp_iso,0.53188,0.01900,-inf,inf,Å²
2,tbti,atom_site,Ti,adp_iso,0.47759,0.03113,-inf,inf,Å²
3,tbti,atom_site,O1,fract_x,0.32803,0.00009,-inf,inf,
4,tbti,atom_site,O1,adp_iso,0.45043,0.01652,-inf,inf,Å²
5,tbti,atom_site,O2,adp_iso,0.23747,0.02588,-inf,inf,Å²
6,heidi,extinction,,radius,26.46071,1.00034,-inf,inf,μm
7,heidi,linked_crystal,,scale,2.92356,0.04884,-inf,inf,


Set fit bounds for all free parameters using `multiplier=1.5`. In this
tutorial that means the posterior pair plot will later refer to a
`±1.5 × uncertainty` region in its title. To widen the sampling window,
increase the multiplier explicitly.

In [21]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty(multiplier=1.5)

Displaying the free parameters again is a convenient way to confirm
that the fit bounds have been assigned as expected before launching the
sampler.

In [22]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,tbti,atom_site,Tb,adp_iso,0.53188,0.01900,0.50338,0.56038,Å²
2,tbti,atom_site,Ti,adp_iso,0.47759,0.03113,0.43090,0.52428,Å²
3,tbti,atom_site,O1,fract_x,0.32803,0.00009,0.32789,0.32817,
4,tbti,atom_site,O1,adp_iso,0.45043,0.01652,0.42565,0.47520,Å²
5,tbti,atom_site,O2,adp_iso,0.23747,0.02588,0.19866,0.27629,Å²
6,heidi,extinction,,radius,26.46071,1.00034,24.96020,27.96123,μm
7,heidi,linked_crystal,,scale,2.92356,0.04884,2.85030,2.99682,


## Step 6: Configure and Run DREAM

We now switch from the local minimizer to the Bayesian DREAM sampler.

The settings below are intentionally small so the tutorial runs
quickly. For production analysis you would usually increase the number
of steps (`steps`) and often the burn-in (`burn`) as well. When
needed, the DREAM API also lets you tune how chains are initialized
through the `init` setting. Other sampler settings such as `thin` and
`pop` can be adjusted as well. The current EasyDiffraction defaults
use `steps=3000`, `init='lhs'`, and `parallel=0`, which tells
BUMPS-DREAM to use all available CPUs for population evaluations.

The `burn` setting is auto-resolved when left unset. Here we override
`steps` with a smaller value to keep the tutorial fast, and the
effective burn-in is recomputed automatically.

In [23]:
project.analysis.minimizer.show_supported()

Minimizer types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [24]:
project.analysis.minimizer.type = 'bumps (dream)'

⚠️ Switching minimizer type removes these settings: max_iterations.                                                               


⚠️ Switching minimizer type adds these settings with defaults: burn_in_steps=600, initialization_method='latin_hypercube',        
   parallel_workers=0, population_size=4, random_seed=None, sampling_steps=3000, thinning_interval=1.                             


Current minimizer changed to


bumps (dream)


In [25]:
project.analysis.minimizer.sampling_steps = 100  # lower than the default 3000
project.analysis.minimizer.burn_in_steps = 20  # lower than the default 600

In [26]:
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'heidi' for 'single' fitting


🚀 Starting fit process with 'bumps (dream)'...


📈 Bayesian sampling progress:


,iteration,progress,time (s),log posterior,phase
1,1/121,,6.56,-1443.27,pre-processing
2,7/121,5.8%,8.50,-1385.12,burn-in
3,14/121,11.6%,10.50,-1373.46,burn-in
4,20/121,16.5%,12.26,-1369.25,burn-in
5,21/121,17.4%,12.73,-1368.89,sampling
6,26/121,21.5%,14.41,-1366.62,sampling
7,31/121,25.6%,15.88,-1365.25,sampling
8,36/121,29.8%,17.34,-1364.07,sampling
9,41/121,33.9%,18.74,-1362.81,sampling
10,46/121,38.0%,20.22,-1361.13,sampling


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Bayesian sampling complete.


⚠️ Convergence diagnostics indicate the posterior may be poorly mixed.                                                            


## Step 7: Inspect Bayesian Results

The fit-results display now includes sampler settings, convergence
diagnostics, committed parameter values, and posterior summary
statistics.

In [27]:
project.display.fit.results()

Bayesian fit results


⚠️ Overall status: completed with warnings


💬 Sampler status: DREAM sampling completed


🧪 Sampler: dream


🎯 Committed point estimate: Best posterior sample


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,median,95% interval,r-hat,ess bulk
1,tbti,atom_site,Tb,adp_iso,Å²,0.5321,"[0.5150, 0.5457]",1.478,57.2
2,tbti,atom_site,Ti,adp_iso,Å²,0.4775,"[0.4576, 0.4992]",1.430,62.8
3,tbti,atom_site,O1,fract_x,,0.3280,"[0.3280, 0.3281]",1.534,53.8
4,tbti,atom_site,O1,adp_iso,Å²,0.4507,"[0.4345, 0.4636]",1.472,57.9
5,tbti,atom_site,O2,adp_iso,Å²,0.2369,"[0.2184, 0.2574]",1.397,64.9
6,heidi,extinction,,radius,μm,26.4258,"[25.6565, 27.1753]",1.455,59.4
7,heidi,linked_crystal,,scale,,2.9235,"[2.8734, 2.9588]",1.434,60.8


⚠️ r-hat > 1.01: Consider longer sampling, better initialization, or reparameterization.                                          


⚠️ ess bulk < 400: Consider longer sampling or reparameterization.                                                                


The correlation and posterior-pair plots are complementary:

- `plot_param_correlations` summarizes pairwise structure in a compact
  matrix.
- `plot_posterior_pairs` shows marginal densities on the diagonal and
  posterior contours off-diagonal. In this tutorial its title also
  reminds you that the display region follows the `±1.5 × uncertainty`
  bounds defined above, while numeric subplot ranges are omitted to
  keep the grid readable.

In [28]:
project.display.fit.correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
project.display.posterior.pairs()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The one-dimensional posterior distributions below make it easier to
inspect individual parameters in isolation, including asymmetry or
multimodality.

In [30]:
project.display.posterior.distribution()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finally, the posterior predictive plot propagates the sampled
parameter uncertainty into the calculated single-crystal reflection
intensities.

In [31]:
project.display.posterior.predictive(expt_name='heidi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>